In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# §6.18 Spin, Magnetic Moments, and the Electron in a Magnetic Field

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.18",
    title="Spin, Magnetic Moments, and the Electron in a Magnetic Field",
    blurb="The electron's hidden turn, made physical. We already know spin as the "
    "smallest angular momentum; here it acquires a body — a magnetic moment twice as "
    "strong as motion would give, an energy in a magnetic field that splits and "
    "precesses the spin, and a fourth quantum number that completes the atom and "
    "fixes the length of every row in the periodic table. And by coupling to the "
    "field of its own orbit, spin sets up the fine splitting of the levels we just "
    "found.",
    difficulty="advanced",
    estimate="160–195 min",
)

## Notebook overview

Movement IV opens by settling an account. Movement I built the *kinematics* of the electron's spin — the
two-state space ([§6.4](stern-gerlach-qubit.ipynb)), the Pauli algebra and uncertainty ([§6.6](pauli-uncertainty.ipynb)), precession and Rabi flopping ([§6.7](time-evolution.ipynb)), the
Bloch sphere ([§6.8](bloch-sphere-entanglement.ipynb)) — and Movement III sharpened it: [§6.14](angular-momentum-algebra.ipynb) derived spin-$\tfrac12$ as the smallest
representation of the rotation algebra, and [§6.15](orbital-angular-momentum.ipynb) showed that orbital motion *cannot* produce
half-integers. What none of that supplied is the **physics**: what spin *does* in the world. That is this
notebook.

The new content is four things, and we take pains not to re-teach what is already built. First, the
electron's **magnetic moment** and its **anomalous $g$-factor**: a spin produces a magnetic moment about
**twice** as strong as the same amount of *orbital* angular momentum would — $g_s\approx2$ against
$g_l=1$ — a fact Schrödinger's theory cannot explain, which falls out exactly of the Dirac equation
($g=2$) and whose tiny excess $g-2$ is the most precisely tested prediction in all of physics (QED).
Second, the **Zeeman energy** $H=-\boldsymbol\mu\cdot\mathbf B$: a magnetic field gives the two spin
states different energies, splitting them by $g_s\mu_B B$ and precessing a tilted spin at the
$g$-corrected **Larmor frequency** — the working principle of NMR, ESR, MRI, and the spin qubit. This is
where the Stern–Gerlach experiment of [§6.4](stern-gerlach-qubit.ipynb) comes full circle: the beam split because the two spin states
have different energies in a field. Third, the **composite state**: an electron is a spatial orbital
*tensored with* a spin, $|n,l,m_l\rangle\otimes|m_s\rangle$, so each hydrogen shell's $n^2$ orbitals
become $2n^2$ states — the factor of two that [§6.17](hydrogen-atom.ipynb) had to *assume*, now **derived**, and the origin of the
periodic table's row lengths. Fourth, the **spin–orbit coupling** $\propto\mathbf L\cdot\mathbf S$: the
electron's spin couples to the magnetic field of its own orbital motion, splitting each level into
total-angular-momentum sublevels — the atomic **fine structure**. Evaluating $\mathbf L\cdot\mathbf S$
turns out to require knowing how to *add* the angular momenta $\mathbf L$ and $\mathbf S$, which is exactly
the subject of the next notebook: §6.18 poses the question that [§6.19](addition-angular-momenta.ipynb) answers.

As in every Volume VI notebook, each exercise opens with a **crystal-clear statement** and enumerated parts, each naming the exact operation — the spin matrices $\mathbf S=\tfrac\hbar2\boldsymbol\sigma$
(from [§6.6](pauli-uncertainty.ipynb)/[§6.14](angular-momentum-algebra.ipynb)), the Zeeman Hamiltonian as a matrix, `numpy.kron` for the spatial$\otimes$spin product,
`scipy.linalg.expm` for precession, `numpy.linalg.eigh` for spectra, and $\mathbf L\cdot\mathbf S=
\tfrac12(J^2-L^2-S^2)$ from the [§6.14](angular-momentum-algebra.ipynb) angular-momentum matrices.

> **Conventions and units.** We set $\hbar=1$ and the Bohr magneton $\mu_B=1$, so energies are measured
> in units of $\mu_B$ and the field $B$ in the corresponding units; the $g$-factors are $g_l=1$
> (orbital) and $g_s\approx2.0023$ (spin). The field is along $z$. The spin operators are $\mathbf S=
> \tfrac\hbar2\boldsymbol\sigma$, i.e. the $j=\tfrac12$ matrices of [§6.14](angular-momentum-algebra.ipynb). **Anti-redundancy:** the Pauli
> algebra ([§6.6](pauli-uncertainty.ipynb)), the Bloch sphere ([§6.8](bloch-sphere-entanglement.ipynb)), and the mechanics of precession/Rabi ([§6.7](time-evolution.ipynb)) are *used and cited*
> here, not re-derived. See Sakurai & Napolitano and Griffiths (spin, magnetic moments, the Zeeman
> effect, spin–orbit coupling); and Notebooks [§6.4](stern-gerlach-qubit.ipynb) (Stern–Gerlach), [§6.6](pauli-uncertainty.ipynb) (Pauli), [§6.7](time-evolution.ipynb) (precession/Rabi),
> [§6.8](bloch-sphere-entanglement.ipynb) (tensor product/entanglement), [§6.14](angular-momentum-algebra.ipynb) (spin from the algebra), [§6.17](hydrogen-atom.ipynb) (the $2n^2$ asserted).

## Theory in brief

### Spin as intrinsic angular momentum (recap, cited)

The electron carries an intrinsic angular momentum $\mathbf S$ with $s=\tfrac12$, living in a
two-dimensional internal space with $\mathbf S=\tfrac\hbar2\boldsymbol\sigma$ ([§6.6](pauli-uncertainty.ipynb), [§6.14](angular-momentum-algebra.ipynb)). It has **no**
position-space wavefunction — it is not "the electron spinning," but a genuine internal degree of freedom
the rotation algebra demands ([§6.14](angular-momentum-algebra.ipynb)) and orbital motion cannot supply ([§6.15](orbital-angular-momentum.ipynb)). We take this as given and
turn to the new physics.

### The magnetic moment and the $g$-factor

A charged particle with angular momentum has a **magnetic moment**. For orbital angular momentum,

```{math}
:label: eq-g-factor
\boldsymbol\mu_L=-g_l\frac{\mu_B}{\hbar}\mathbf L\ (g_l=1),\qquad \boldsymbol\mu_S=-g_s\frac{\mu_B}{\hbar}\mathbf S\ (g_s\approx2.0023) ,
```

with $\mu_B$ the **Bohr magneton**. The spin's $g_s\approx2$ means a spin produces **twice** the magnetic
moment per unit angular momentum that orbital motion does — an anomaly Schrödinger's theory cannot
explain. It emerges exactly from the **Dirac equation** ($g=2$), and the excess $g-2\approx0.00232$ is a
triumph of **quantum electrodynamics** (both are horizons here).

### The Zeeman Hamiltonian and level splitting

In a field $\mathbf B=B\hat z$, the spin energy is

```{math}
:label: eq-zeeman
H=-\boldsymbol\mu_S\cdot\mathbf B=g_s\frac{\mu_B}{\hbar}B\,S_z=\tfrac12 g_s\mu_B B\,\sigma_z ,
```

with eigenstates spin-up/down split by $\Delta E=g_s\mu_B B$. This **Zeeman** splitting is the energy
behind the two Stern–Gerlach beams ([§6.4](stern-gerlach-qubit.ipynb), full circle): the beam split because the two spin states have
different energies in a field. (The full atomic Zeeman effect adds the orbital moment, $g_l\mathbf L+g_s
\mathbf S$.)

### Larmor precession at the $g$-corrected frequency

A spin tilted from $\mathbf B$ precesses about it ([§6.7](time-evolution.ipynb)) at the **Larmor frequency**

```{math}
:label: eq-larmor-g
\omega_L=\frac{g_s\mu_B B}{\hbar},\qquad \langle S_x\rangle(t)=\tfrac12\cos\omega_L t ,
```

now with the $g$-factor made physical. This is the resonance frequency probed by **NMR**, **ESR/EPR**,
and **MRI**, and the frequency at which a spin qubit is driven. The precession itself is the [§6.7](time-evolution.ipynb) result;
what is new is that $\omega_L$ is a measurable property through $g$.

### The composite state: spatial $\otimes$ spin

The full electron state is the **tensor product** of its spatial and spin parts,

```{math}
:label: eq-spatial-spin
|\Psi\rangle=|n,l,m_l\rangle\otimes|m_s\rangle,\qquad \text{each shell: } n^2\times 2=2n^2\ \text{states} ,
```

labelled by $n,l,m_l,m_s=\pm\tfrac12$ (`numpy.kron`). Spin's factor-two degeneracy, independent of the
spatial state, turns hydrogen's $n^2$ orbitals ([§6.17](hydrogen-atom.ipynb)) into $2n^2$ states — $2,8,18,32$, the factor of two
now **derived**. This is the origin of the periodic table's rows (filled by the Pauli exclusion principle
of [§6.20](identical-particles.ipynb)). A spatial–spin state can be a product or, in general, **entangled** ([§6.8](bloch-sphere-entanglement.ipynb)) — and spin–orbit
coupling entangles them.

### Spin–orbit coupling: the bridge

In the atom's rest frame the electron sees the nucleus orbiting it, a current loop whose magnetic field
couples to the spin — the **spin–orbit** interaction $H_{SO}\propto\mathbf L\cdot\mathbf S$. With
$\mathbf J=\mathbf L+\mathbf S$,

```{math}
:label: eq-spin-orbit
\mathbf L\cdot\mathbf S=\tfrac12\!\left(J^2-L^2-S^2\right),\qquad \langle\mathbf L\cdot\mathbf S\rangle_j=\tfrac{\hbar^2}{2}\big[j(j+1)-l(l+1)-s(s+1)\big] ,
```

so spin–orbit coupling **splits** each orbital level into distinct-$j$ sublevels (an $l=1$ level splits
into $j=\tfrac32$ and $j=\tfrac12$) — the **fine structure** of atomic spectra (the sodium D-line
doublet), computed perturbatively in [§6.21](perturbation-fine-structure.ipynb). But evaluating $\mathbf L\cdot\mathbf S$ requires the
eigenstates of $J^2=(\mathbf L+\mathbf S)^2$ — i.e. how to **add** $\mathbf L$ and $\mathbf S$. That is
exactly the next notebook ([§6.19](addition-angular-momenta.ipynb)).

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import expm

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED, BLUE = "#c1121f", "#1d4e89"

HBAR = 1.0
MU_B = 1.0  # Bohr magneton (energies measured in units of μ_B)
G_L = 1.0  # orbital g-factor
G_S = 2.0023193  # electron spin g-factor (Dirac g=2 plus the QED anomaly)


def angular_momentum_matrices(j, hbar=HBAR):
    r"""The angular-momentum matrices $J_x,J_y,J_z,J^2,J_+,J_-$ for a given $j$ — reused from §6.14.

    Built from the matrix elements $J_\pm|j,m\rangle=\hbar\sqrt{j(j+1)-m(m\pm1)}|j,m\pm1\rangle$ in the
    descending-$m$ basis; $s=\tfrac12$ gives the spin operators $\mathbf S=\tfrac\hbar2\boldsymbol\sigma$.
    """
    dim = int(round(2 * j + 1))
    m = np.arange(j, -j - 1, -1.0)
    Jz = hbar * np.diag(m).astype(complex)
    Jp = np.zeros((dim, dim), dtype=complex)
    Jm = np.zeros((dim, dim), dtype=complex)
    for a in range(dim):
        ma = m[a]
        cp = j * (j + 1) - ma * (ma + 1)
        if cp > 1e-12:
            Jp[a - 1, a] = hbar * np.sqrt(cp)
        cm = j * (j + 1) - ma * (ma - 1)
        if cm > 1e-12:
            Jm[a + 1, a] = hbar * np.sqrt(cm)
    Jx = (Jp + Jm) / 2
    Jy = (Jp - Jm) / 2j
    J2 = Jx @ Jx + Jy @ Jy + Jz @ Jz
    return Jx, Jy, Jz, J2, Jp, Jm


# the spin-½ operators S = (ℏ/2)σ (from §6.6/§6.14)
SX, SY, SZ, S2, SP, SM = angular_momentum_matrices(0.5)


def zeeman_hamiltonian(B, g):
    r"""The Zeeman Hamiltonian $H=g\mu_B B\,S_z/\hbar=\tfrac12 g\mu_B B\,\sigma_z$ for a spin in a field $B\hat z$ {eq}`eq-zeeman`."""
    return g * MU_B * B * SZ / HBAR


def magnetic_moment(g):
    r"""The spin magnetic-moment operators $\boldsymbol\mu=-g(\mu_B/\hbar)\mathbf S$ {eq}`eq-g-factor`.

    Returns $(\mu_x,\mu_y,\mu_z)$; the gyromagnetic ratio is $\gamma=g\mu_B/\hbar$.
    """
    scale = -g * MU_B / HBAR
    return scale * SX, scale * SY, scale * SZ


def larmor_frequency(B, g):
    r"""The Larmor precession frequency $\omega_L=g\mu_B B/\hbar$ {eq}`eq-larmor-g`."""
    return g * MU_B * B / HBAR


def spin_orbit_LS(l, s=0.5):
    r"""The spin–orbit operator $\mathbf L\cdot\mathbf S=L_xS_x+L_yS_y+L_zS_z$ on the orbital$\otimes$spin space {eq}`eq-spin-orbit`.

    Built with ``numpy.kron`` from the §6.14 matrices ($j=l$ for $\mathbf L$, $j=s$ for $\mathbf S$); its
    distinct eigenvalues are the $j$-multiplet values $\tfrac12[j(j+1)-l(l+1)-s(s+1)]$ (with $\hbar=1$).
    """
    Lx, Ly, Lz, _, _, _ = angular_momentum_matrices(l)
    sx, sy, sz, _, _, _ = angular_momentum_matrices(s)
    return np.kron(Lx, sx) + np.kron(Ly, sy) + np.kron(Lz, sz)

## Exercise 1 — The magnetic moment and the $g$-factor

Compare the magnetic moment produced by spin with that produced by orbital angular momentum, and
show spin's is about **twice** as strong per unit angular momentum — the anomalous $g_s\approx2$
{eq}`eq-g-factor`.

1. Write the moments $\boldsymbol\mu_L=-g_l(\mu_B/\hbar)\mathbf L$ ($g_l=1$) and
   $\boldsymbol\mu_S=-g_s(\mu_B/\hbar)\mathbf S$ ($g_s\approx2$).
2. The **gyromagnetic ratio** is $\gamma= g\mu_B/\hbar$ — the moment per unit angular momentum.
3. Compute $\gamma_S/\gamma_L=g_s/g_l\approx2$: spin gives twice the moment.
4. Note $g_s=2$ is the Dirac prediction and $g-2$ the QED anomaly. Frame: the electron's anomalous
   magnetic moment — genuinely new physics.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    np.array([G_S, G_L]),
    np.array([2.0, 1.0]),
    "the electron spin g-factor is ≈2, twice the orbital g-factor — spin's anomalous magnetic moment",
    atol=1e-2,
)

## Exercise 2 — The Zeeman Hamiltonian and level splitting

Find the energy splitting of a spin in a magnetic field, confirming the **Zeeman** splitting
$\Delta E=g_s\mu_B B$ {eq}`eq-zeeman`.

1. Build $H=\tfrac12 g_s\mu_B B\,\sigma_z$ (the `zeeman_hamiltonian` helper).
2. Diagonalize with `numpy.linalg.eigh`.
3. Confirm the two spin states split by $\Delta E=g_s\mu_B B$.
4. Plot the splitting against $B$ (linear). Frame: the Zeeman energy — *why* the Stern–Gerlach
   beam split ([§6.4](stern-gerlach-qubit.ipynb), full circle).

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
B_test = 1.5
E_test = np.linalg.eigvalsh(zeeman_hamiltonian(B_test, G_S))
validate.close(
    E_test[-1] - E_test[0],
    G_S * MU_B * B_test,
    "the Zeeman splitting is g μ_B B",
    rtol=1e-6,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 3 — Larmor precession at the $g$-corrected frequency

Precess a spin in a field and identify the **Larmor frequency** $\omega_L=g_s\mu_B B/\hbar$
{eq}`eq-larmor-g`, confirming $\langle S_x\rangle(t)=\tfrac12\cos\omega_L t$.

1. Start a spin along $x$ (the state $(|\!\uparrow\rangle+|\!\downarrow\rangle)/\sqrt2$).
2. Evolve it under the Zeeman $H$ with `scipy.linalg.expm` (the time-evolution method of [§6.7](time-evolution.ipynb)).
3. Compute $\langle S_x\rangle(t)$ and confirm it is $\tfrac12\cos\omega_L t$.
4. Identify $\omega_L=g_s \mu_B B/\hbar$. Frame: the precession of [§6.7](time-evolution.ipynb), now with the physical
   $g$-factor — the magnetic-resonance frequency.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    larmor_ok,
    "the spin precesses at the Larmor frequency ω_L = g μ_B B/ℏ, with ⟨S_x⟩(t)=½cos(ω_L t)",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4 — The composite state: spatial $\otimes$ spin

Build the full electron state as an orbital$\otimes$spin product with `numpy.kron`, and show that
spatial operators and spin operators act on their own factors {eq}`eq-spatial-spin`.

1. Take an orbital state (a unit vector in the $(2l+1)$-dimensional $l$-space) and a spin state
   $|m_s\rangle$.
2. Form the tensor product $|\Psi\rangle=|\text{orbital}\rangle\otimes|\text{spin} \rangle$ with
   `numpy.kron`; its dimension is $(2l+1)\times2$.
3. Verify a **spin** operator acts as $I_{\text{orb}}\otimes S_z$ and an **orbital** operator as
   $L_z\otimes I_{\text{spin}}$.
4. Confirm the two commute — they act on independent factors. Frame: the electron's state has a
   spatial factor and a spin factor.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    tensor_ok,
    "the electron's state is orbital⊗spin (numpy.kron): it carries n,l,m_l,m_s and orbital/spin operators act on independent, commuting factors",
)

## Exercise 5 — The $2n^2$ counting: completing hydrogen

Count the states in each hydrogen shell *including spin*, and confirm $2n^2=2,8,18,32$ — the
factor of two that [§6.17](hydrogen-atom.ipynb) asserted, now derived from spin {eq}`eq-spatial-spin`.

1. Recall the $n^2$ spatial orbitals per shell ([§6.17](hydrogen-atom.ipynb)).
2. Multiply by the 2 spin states.
3. Obtain $2n^2=2,8,18,32$.
4. Note this is the derived factor of two, and (with the Pauli exclusion principle of [§6.20](identical-particles.ipynb)) the
   periodic-table row lengths. Frame: spin completes the atom.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.check(
    counts_ok,
    "each hydrogen shell holds 2n² states = 2,8,18,32 — spin supplies the factor of two completing hydrogen's degeneracy",
)

## Exercise 6 — Spin–orbit coupling $\mathbf L\cdot\mathbf S$

Evaluate the spin–orbit operator $\mathbf L\cdot\mathbf S=\tfrac12(J^2-L^2-S^2)$ and find how it
splits an orbital level into total-angular-momentum sublevels {eq}`eq-spin-orbit`.

1. Build $\mathbf L\cdot\mathbf S=L_xS_x+L_yS_y+L_zS_z$ on the orbital$\otimes$spin space (the
   `spin_orbit_LS` helper, `numpy.kron` of the [§6.14](angular-momentum-algebra.ipynb) matrices).
2. For $l=1$, $s=\tfrac12$, note $j$ can be $\tfrac32$ or $\tfrac12$.
3. Diagonalize with `numpy.linalg.eigh` and compare the distinct eigenvalues to
   $\tfrac12[j(j+1)-l(l+1)-s(s+1)]$.
4. Confirm the level splits into a $j=\tfrac32$ and a $j=\tfrac12$ sublevel — the fine-structure
   doublet. Frame: spin–orbit coupling splits the levels — atomic fine structure — and needs the
   addition of $\mathbf L$ and $\mathbf S$ ([§6.19](addition-angular-momenta.ipynb)).

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
l_chk = 1
validate.close(
    np.array(sorted(set(np.round(np.linalg.eigvalsh(spin_orbit_LS(l_chk, s)), 6)))),
    np.array(
        sorted(
            0.5 * (j * (j + 1) - l_chk * (l_chk + 1) - s * (s + 1))
            for j in [l_chk + s, l_chk - s]
        )
    ),
    "L·S = ½(J²−L²−S²) splits a level into j-multiplets ½[j(j+1)−l(l+1)−s(s+1)] — the origin of fine structure",
    rtol=1e-6,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — The electron-spin qubit in a field *(student)*

Model a single electron spin as a qubit driven by a static plus an oscillating field, and show it
undergoes **resonant** transitions at the Larmor frequency — the principle of magnetic resonance
and spin-qubit control {eq}`eq-zeeman`, {eq}`eq-larmor-g`.

1. A static field $B\hat z$ sets the qubit gap $\hbar\omega_L$ (the Zeeman splitting).
2. Add a weak transverse oscillating field $B_1\cos(\omega_d t)\hat x$ and evolve the spin
   (`scipy.linalg.expm`, time-stepped), starting from spin-down.
3. Scan the drive frequency $\omega_d$ and record the spin-flip probability.
4. Show it **peaks sharply at $\omega_d=\omega_L$** — resonance. Frame: the spin as a controllable
   qubit — Movement I's kinematics meeting this notebook's energetics.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    resonance_ok,
    "an electron spin in a field is a qubit addressed at its Larmor frequency: the driven spin flips resonantly, peaking sharply at ω_d=ω_L",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 8 — The electron, completed

Movement I taught us how a spin *behaves* — its states, its algebra, its precession, its sphere. This
notebook gave that spin a physical **body**: a magnetic moment twice as strong as motion would grant
($g_s\approx2$, a Dirac/QED fact), an energy in a field that splits and turns it (the Zeeman effect and
Larmor precession, the engine of magnetic resonance), a place in the atom that doubles every shell and
sets the periodic table's rhythm ($2n^2$, the factor of two now derived), and a coupling to its own orbit
that will split the spectral lines ($\mathbf L\cdot\mathbf S$).

**This exercise (synthesis).** No new computation: the completed electron is the result. It is now fully
described — a spatial wavefunction *and* a spin, four quantum numbers, one particle. But the spin–orbit
coupling posed a question we cannot yet answer: to evaluate $\mathbf L\cdot\mathbf S$ we had to
diagonalize $J^2=(\mathbf L+\mathbf S)^2$ — we needed the eigenstates of the *sum* of two angular momenta.
The next notebook ([§6.19](addition-angular-momenta.ipynb)) builds exactly that machinery, the **addition of angular momenta** and the
**Clebsch–Gordan coefficients**, the last tool required before fine structure ([§6.21](perturbation-fine-structure.ipynb)) and the
many-electron atom ([§6.20](identical-particles.ipynb)).

Notice how the number **two** keeps appearing: two spin states, twice the magnetic moment, $2n^2$
electrons per shell, a $g$-factor of $2$. It is the same two each time — the dimension of the smallest
nonzero angular momentum, the $j=\tfrac12$ of [§6.14](angular-momentum-algebra.ipynb) — propagating outward from an algebra into the
magnetism of matter and the length of a row of the periodic table.

## Notebook summary

The physics of spin — the opening of Movement IV.

- **The $g$-factor** {eq}`eq-g-factor`: spin's magnetic moment is $g_s\approx2$ times $\mu_B$ per unit
  angular momentum — twice the orbital $g_l=1$ (Dirac's $g=2$ plus the QED anomaly $g-2$).
- **The Zeeman effect** {eq}`eq-zeeman`: $H=\tfrac12 g_s\mu_B B\,\sigma_z$ splits the spin states by
  $\Delta E=g_s\mu_B B$ — the energy behind the Stern–Gerlach beams ([§6.4](stern-gerlach-qubit.ipynb)).
- **Larmor precession** {eq}`eq-larmor-g`: a tilted spin precesses at $\omega_L=g_s\mu_B B/\hbar$
  (`scipy.linalg.expm`) — the frequency of NMR, ESR, and MRI.
- **Spatial $\otimes$ spin** {eq}`eq-spatial-spin`: the electron state is $|n,l,m_l\rangle\otimes|m_s
  \rangle$ (`numpy.kron`); spin's factor of two makes hydrogen's shells hold $2n^2=2,8,18,32$ states.
- **Spin–orbit coupling** {eq}`eq-spin-orbit`: $\mathbf L\cdot\mathbf S=\tfrac12(J^2-L^2-S^2)$ splits a
  level into $j=l\pm\tfrac12$ sublevels — fine structure — and requires adding $\mathbf L$ and $\mathbf S$.

The electron now has a body and a place in the atom. What remains is to learn to add its two angular
momenta — the subject of the next notebook.

## Outlook

- **The addition of angular momenta and Clebsch–Gordan coefficients ([§6.19](addition-angular-momenta.ipynb))**: the eigenstates of
  $\mathbf J=\mathbf L+\mathbf S$, needed to evaluate $\mathbf L\cdot\mathbf S$.
- **Identical particles and the Pauli exclusion principle ([§6.20](identical-particles.ipynb))**: filling the $2n^2$ states to build
  atoms and the periodic table.
- **Fine structure ([§6.21](perturbation-fine-structure.ipynb))**: the spin–orbit and relativistic corrections by perturbation theory; the
  anomalous Zeeman effect.
- **The electron's $g-2$ and QED; the Dirac equation's $g=2$** (horizons — the deep origin of the
  anomalous moment).
- **Cross-reference** [§6.4](stern-gerlach-qubit.ipynb) (Stern–Gerlach), [§6.6](pauli-uncertainty.ipynb) (Pauli), [§6.7](time-evolution.ipynb) (precession/Rabi), [§6.8](bloch-sphere-entanglement.ipynb) (tensor
  product/entanglement), [§6.14](angular-momentum-algebra.ipynb) (spin from the algebra), [§6.17](hydrogen-atom.ipynb) (the $2n^2$), and forward to [§6.19](addition-angular-momenta.ipynb), [§6.20](identical-particles.ipynb), [§6.21](perturbation-fine-structure.ipynb).

In [ ]:
from ecp.style import footer

footer()